# ⚽ Player Similarity Notebook

**Version:** MVP v0.1.0

Find statistically similar football players using FBref player statistics.

---

## Objective

This notebook allows you to:

- Load a player dataset
- Clean and validate the data
- Find statistically similar players
- Compare players using radar charts

---

## Workflow

1. Configuration
2. Load Dataset
3. Data Validation
4. Data Cleaning
5. Feature Selection
6. Player Similarity
7. Radar Chart
8. Export Results

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

## 1. Configuration

In [2]:
from src.config import DATA_RAW
from src.data_loader import load_dataset
from src.validation import validate_dataset

## 2. Load Dataset

In [3]:
dataset_path = DATA_RAW / "players_data_light-2024_2025.csv"

df = load_dataset(dataset_path)

validate_dataset(df)

df.head()

✓ Dataset validation successful.
Players: 2854
Required columns: OK
Similarity features: 7
Ready for similarity analysis.


,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,...,Att (GK),Thr,Launch%,AvgLen,Opp,Stp,Stp%,#OPA,#OPA/90,AvgDist
0,1,Max Aarons,eng ENG,DF,Bournemouth,eng Premier League,24.0,2000.0,3,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Max Aarons,eng ENG,"DF,MF",Valencia,es La Liga,24.0,2000.0,4,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Rodrigo Abajas,es ESP,DF,Valencia,es La Liga,21.0,2003.0,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,James Abankwah,ie IRL,"DF,MF",Udinese,it Serie A,20.0,2004.0,6,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Keyliane Abdallah,fr FRA,FW,Marseille,fr Ligue 1,18.0,2006.0,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
df[["Player", "Min", "90s", "Gls", "Ast", "xG", "xAG", "PrgC", "PrgP", "PrgR"]].head(10)

,Player,Min,90s,Gls,Ast,xG,xAG,PrgC,PrgP,PrgR
0,Max Aarons,86,1.0,0,0,0.0,0.0,1,8,3
1,Max Aarons,120,1.3,0,0,0.0,0.0,0,6,10
2,Rodrigo Abajas,65,0.7,0,0,0.1,0.0,3,2,3
3,James Abankwah,88,1.0,0,0,0.1,0.0,3,4,1
4,Keyliane Abdallah,3,0.0,0,0,0.0,0.0,1,0,0
5,Yunis Abdelhamid,1033,11.5,0,0,0.2,0.1,4,22,3
6,Himad Abdelli,2842,31.6,6,1,6.4,3.2,107,207,111
7,Mohamed Abdelmoneim,855,9.5,0,0,0.0,0.0,6,52,5
8,Ali Abdi,1393,15.5,5,2,4.3,1.9,35,42,101
9,Saud Abdulhamid,205,2.3,0,1,0.0,0.2,6,9,26


In [5]:
df[["Gls", "Ast", "xG", "xAG", "PrgC", "PrgP", "PrgR"]].describe()

,Gls,Ast,xG,xAG,PrgC,PrgP,PrgR
count,2854.000000,2854.000000,2854.000000,2854.000000,2854.000000,2854.000000,2854.000000
mean,1.682901,1.200771,1.706903,1.215662,20.733006,45.221093,44.796776
std,3.152732,1.946170,2.817612,1.686875,26.635816,50.147121,59.818947
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.100000,0.100000,2.000000,5.000000,3.000000
50%,0.000000,0.000000,0.700000,0.600000,11.000000,28.500000,20.000000
75%,2.000000,2.000000,2.100000,1.600000,29.000000,69.000000,66.000000
max,31.000000,18.000000,27.100000,14.200000,213.000000,362.000000,488.000000


In [6]:
from src.preprocessing import prepare_similarity_data

df_prepared = prepare_similarity_data(df)

print(f"Original players: {len(df)}")
print(f"Prepared players: {len(df_prepared)}")

Original players: 2854
Prepared players: 1570


In [7]:
per90_columns = [
    "Gls_per90",
    "Ast_per90",
    "xG_per90",
    "xAG_per90",
    "PrgC_per90",
    "PrgP_per90",
    "PrgR_per90",
]

df_prepared[
    ["Player", "Squad", "Min", "90s"] + per90_columns
].head()

,Player,Squad,Min,90s,Gls_per90,Ast_per90,xG_per90,xAG_per90,PrgC_per90,PrgP_per90,PrgR_per90
0,Yunis Abdelhamid,Saint-Étienne,1033,11.5,0.000000,0.000000,0.017391,0.008696,0.347826,1.913043,0.260870
1,Himad Abdelli,Angers,2842,31.6,0.189873,0.031646,0.202532,0.101266,3.386076,6.550633,3.512658
2,Ali Abdi,Nice,1393,15.5,0.322581,0.129032,0.277419,0.122581,2.258065,2.709677,6.516129
3,Abel,Osasuna,2074,23.0,0.086957,0.000000,0.021739,0.043478,2.173913,3.347826,4.000000
4,Matthis Abline,Nantes,2768,30.8,0.292208,0.064935,0.275974,0.123377,2.435065,1.558442,5.487013


In [8]:
df_prepared[per90_columns].isna().sum()

Gls_per90     0
Ast_per90     0
xG_per90      0
xAG_per90     0
PrgC_per90    0
PrgP_per90    0
PrgR_per90    0
dtype: int64

In [9]:
df_prepared[
    ["Player", "Squad", "Pos"]
].sample(10)

,Player,Squad,Pos
515,Lutsharel Geertruida,RB Leipzig,DF
355,Liam Delap,Ipswich Town,FW
1085,Ivan Ordets,Bochum,DF
512,Mory Gbane,Reims,"DF,MF"
1427,Tiago Tomás,Wolfsburg,"FW,MF"
329,Diogo Dalot,Manchester Utd,DF
149,Fran Beltrán,Celta Vigo,MF
933,Dwight McNeil,Everton,"MF,FW"
932,Mark McKenzie,Toulouse,DF
543,Mathieu Gorgelin,Le Havre,GK


In [10]:
from src.similarity import find_similar_players

similar_players = find_similar_players(
    df=df_prepared,
    player_name="Leandro Trossard",
    top_n=10,
)

similar_players

,Player,Squad,Pos,Age,Min,Similarity
0,Bradley Barcola,Paris S-G,FW,21.0,2181,0.987821
1,Son Heung-min,Tottenham,FW,32.0,2110,0.983220
2,Bukayo Saka,Arsenal,"FW,MF",22.0,1729,0.981136
3,Álex Berenguer,Athletic Club,"FW,MF",29.0,2339,0.979245
4,Ansgar Knauff,Eint Frankfurt,"MF,DF",22.0,1597,0.973626
5,Rafael Leão,Milan,FW,25.0,2323,0.973626
6,Michael Olise,Bayern Munich,"FW,MF",22.0,2334,0.969274
7,Karim Adeyemi,Dortmund,"FW,MF",22.0,1433,0.968039
8,Serge Gnabry,Bayern Munich,"FW,MF",29.0,1242,0.966996
9,Iñaki Williams,Athletic Club,FW,30.0,2652,0.964187


In [11]:
df_prepared[
    df_prepared["Player"].duplicated(keep=False)
][["Player", "Squad", "Pos", "Min"]].sort_values("Player").head(20)

,Player,Squad,Pos,Min
548,Amine Gouiri,Rennes,"MF,FW",1037
547,Amine Gouiri,Marseille,"FW,MF",1048
1229,Anthony Rouault,Rennes,DF,972
1230,Anthony Rouault,Stuttgart,DF,1212
238,Antonio Candela,Venezia,DF,1008
239,Antonio Candela,Valladolid,DF,906
1264,Brice Samba,Lens,GK,1350
1263,Brice Samba,Rennes,GK,1529
20,Emmanuel Agbadou,Wolves,DF,1410
21,Emmanuel Agbadou,Reims,DF,1260


In [12]:
find_similar_players(
    df=df_prepared,
    player_name="Bukayo Saka",
)

,Player,Squad,Pos,Age,Min,Similarity
0,Son Heung-min,Tottenham,FW,32.0,2110,0.989895
1,Nicolas Pépé,Villarreal,"FW,MF",29.0,1500,0.986096
2,Ansgar Knauff,Eint Frankfurt,"MF,DF",22.0,1597,0.981741
3,Leandro Trossard,Arsenal,FW,29.0,2546,0.981136
4,Franck Honorat,Gladbach,FW,27.0,1439,0.978294
5,Rafael Leão,Milan,FW,25.0,2323,0.976503
6,Lamine Yamal,Barcelona,FW,17.0,2856,0.976486
7,Iñaki Williams,Athletic Club,FW,30.0,2652,0.973960
8,Álex Berenguer,Athletic Club,"FW,MF",29.0,2339,0.973457
9,Michael Olise,Bayern Munich,"FW,MF",22.0,2334,0.970150


In [13]:
# Jugador duplicado sin equipo
find_similar_players(
    df=df_prepared,
    player_name="Omar Marmoush",
)

ValueError: Multiple records found for Omar Marmoush. Specify squad. Available squads: Manchester City, Eint Frankfurt

In [14]:
# Jugador duplicado con equipo
find_similar_players(
    df=df_prepared,
    player_name="Omar Marmoush",
    squad="Manchester City",
)

,Player,Squad,Pos,Age,Min,Similarity
0,Giacomo Raspadori,Napoli,"FW,MF",24.0,1103,0.948734
1,Nick Woltemade,Stuttgart,"FW,MF",22.0,1622,0.947429
2,Oliver Burke,Werder Bremen,FW,27.0,908,0.939595
3,Albert Guðmundsson,Fiorentina,"FW,MF",27.0,1273,0.935000
4,Marco Grüll,Werder Bremen,"FW,MF",26.0,1181,0.927763
5,Mika Biereth,Monaco,FW,21.0,1228,0.924319
6,Kylian Mbappé,Real Madrid,FW,25.0,2907,0.918585
7,Cucho,Betis,FW,25.0,1139,0.918281
8,Julián Álvarez,Atlético Madrid,FW,24.0,2509,0.912285
9,Kevin Schade,Brentford,FW,22.0,2293,0.910641
